In [1]:
import pandas as pd

import os


### create functional dataset

In [3]:
df = pd.read_csv("/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/preprocessing/tests/process_claras_dataset/csv/proto_dataset_multimodal.csv")
# participant_id,session,age,dataset,dx,T1aff_path,T1_synthseg_path,T2aff_path,T2_synthseg_path,PDaff_path,PD_synthseg_path,FLAIRaff_path,FLAIR_synthseg_path,has_T2,has_PD,has_FLAIR,split
# sid,paired,resolution,modality,split,iid,org_img_path,seg_synthseg_path,seg_supersynth_path,latent_path,latent_seg_synthseg_merged_3_path,latent_normalized_wm_path,latent_seg_supersynth_merged_3_path,latent_seg_supersynth_merged_8_path,synthsr_path,latent_synthsr_path

# mapping_columns = {
#     "sid": "participant_id",
    
# remove from the df PDaff_path,PD_synthseg_path,has_PD
df = df.drop(columns=["PDaff_path", "PD_synthseg_path", "has_PD"])
# remove from the dataset the rows that have no T2 or FLAIR
df = df[(df["has_T2"] == 1) | (df["has_FLAIR"] == 1)]

# select only one session per subject, the session should be the one with the most modalities available, if there is a tie, select the first one
df = df.sort_values(by=["participant_id", "has_T2", "has_FLAIR"], ascending=[True, False, False])
df = df.drop_duplicates(subset=["participant_id"], keep="first")

new_df = []

def add_row(row, modality):
    new_row = {
        "sid": row["participant_id"],
        "paired": 0,
        "resolution": 3,
        "modality": f"{modality}W" if modality in ["T1", "T2"] else f"T2{modality}",
        "split": "train",
        "iid": os.path.basename(row[f"{modality}aff_path"]).replace(".nii.gz", ""),
        "raw_org_img_path": row[f"{modality}aff_path"],
        "raw_seg_synthseg_path": row[f"{modality}_synthseg_path"],
        # "raw_seg_supersynth_path": row[f"{modality}_synthseg_path"],
    }
    return new_row

for index, row in df.iterrows():
    # add the T1W modality
    _new_row = add_row(row, "T1")
    new_df.append(_new_row)
    # add the T2W modality if it exists
    if row["has_T2"] == 1:
        _new_row =add_row(row, "T2")
        new_df.append(_new_row)
    # add the FLAIR modality if it exists
    if row["has_FLAIR"] == 1:
        _new_row = add_row(row, "FLAIR")
        new_df.append(_new_row)

display_df = pd.DataFrame(new_df)
output_path = "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/preprocessing/tests/process_claras_dataset/csv/claras_data.csv"
display_df.to_csv(output_path, index=False)

In [ ]:
# import os
# import numpy as np
# from tqdm import tqdm
# import pandas as pd
# import random
# import sys


# # pytorch
# sys.path.append('/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/utils')


# import perp_segmentation_and_resampling as perp_segmentation_and_resampling 


# # def segment_and_resample(pred_img_path, save_path_name, verify=True, algorithm="synthseg"):

# pred_img_path = "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/zip/training/unzip/release_20260414/Training_prospective/T1W/3T/P_T1W_3T_0006.nii.gz"
# save_path_name = "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/mni/P_T1W_3T_0006_seg.nii.gz"
# perp_segmentation_and_resampling.segment_and_resample(
#     pred_img_path,
#     save_path_name,
#     verify=True,
#     cortical_parcelation=True
# )


### Merge with training dataset

In [4]:
calra_df = pd.read_csv("/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/csv/claras_data.csv")
train_df = pd.read_csv("/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/csv/train_data.csv")


# find similar columns between the two dataframes
similar_columns = list(set(calra_df.columns).intersection(set(train_df.columns)))
# order the columns as in the train_df
similar_columns = [col for col in train_df.columns if col in similar_columns]
# concatenate the two dataframes using only the similar columns
combined_df = pd.concat([train_df[similar_columns], calra_df[similar_columns]], ignore_index=True)
# save the combined dataframe to a csv file
combined_df.to_csv("/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/csv/train_data_combined.csv", index=False)